# 04 — Coaching Demo
End-to-end pipeline: load one of your own games → run all models → show coaching output.

Before running:
1. Add your PUUID to `.env` as `MY_PUUID`
2. Run `python src/collect_pro.py` to fetch pro data
3. Optionally run `python src/live_capture.py` during a game and note the session filename

In [ ]:
import sys, os
sys.path.insert(0, '../src')
from dotenv import load_dotenv
load_dotenv('../.env')

MY_PUUID = os.getenv('MY_PUUID', '')
if not MY_PUUID:
    print('Set MY_PUUID in .env')
else:
    print(f'PUUID loaded: {MY_PUUID[:12]}…')

In [ ]:
# Fetch your recent match IDs
from collect_pro import get_match_ids, fetch_match_cached

my_matches = get_match_ids(MY_PUUID, count=5)
print('Your 5 most recent TFT matches:')
for i, m in enumerate(my_matches):
    print(f'  [{i}] {m}')

In [ ]:
# Pick a match to analyze (change index as needed)
MATCH_INDEX = 0
match_id = my_matches[MATCH_INDEX]
print(f'Fetching match: {match_id}')
match_data = fetch_match_cached(match_id)

# Extract your participant record
import json
from features import extract_participant_features

participants = match_data['info']['participants']
me = next((p for p in participants if p.get('puuid') == MY_PUUID), None)
if me:
    feats = extract_participant_features(me)
    print(f"Your placement: {feats['placement']}")
    print(f"Final level: {feats['level']} | Gold left: {feats['gold_left']} | Last round: {feats['last_round']}")
    print(f"Top trait: {feats['top_trait']} | Augments: {feats['augment_1']}, {feats['augment_2']}, {feats['augment_3']}")
else:
    print('Your PUUID was not found in this match.')

In [ ]:
# Run the full coaching engine
from coach import analyze_match
from pathlib import Path

# Optional: point to a live session file if you captured one
SESSION_FILE = None  # e.g. Path('../data/live/session_20250101_120000.json')

analyze_match(match_id, MY_PUUID, session_path=SESSION_FILE)

In [ ]:
# Show how your augments compared to pro win rates for your comp
import pandas as pd
aug_df = pd.read_csv('../data/processed/augment_winrates.csv')
top_trait = feats['top_trait']
comp_augs = aug_df[aug_df['top_trait'] == top_trait].head(10)
print(f'\nTop augments for {top_trait} comps in Challenger:')
print(comp_augs[['augment','games','top4_rate','avg_placement']].to_string(index=False))

# Highlight your picks
your_augs = [feats['augment_1'], feats['augment_2'], feats['augment_3']]
print(f'\nYour augments: {your_augs}')
for aug in your_augs:
    row = comp_augs[comp_augs['augment'] == aug]
    if not row.empty:
        print(f'  {aug}: {row["top4_rate"].values[0]:.0%} top-4 (rank {comp_augs.index.get_loc(row.index[0])+1} for this comp)')
    else:
        print(f'  {aug}: no data for this comp')

In [ ]:
# Show item recommendations for your units
item_df = pd.read_csv('../data/processed/item_winrates.csv')
unit_items = json.loads(feats['_unit_items'])

print('Item analysis per unit:\n')
for unit_id, items in unit_items.items():
    if not items:
        continue
    unit_rows = item_df[item_df['unit_id'] == unit_id]
    if unit_rows.empty:
        continue
    user_combo = str(tuple(sorted(items)))
    best = unit_rows.iloc[0]
    user_row = unit_rows[unit_rows['item_combo'] == user_combo]
    user_rate = user_row['top4_rate'].values[0] if not user_row.empty else None
    
    print(f'{unit_id}:')
    print(f'  Your build: {user_combo} → {f"{user_rate:.0%} top-4" if user_rate else "no pro data"}')
    print(f'  Best build: {best["item_combo"]} → {best["top4_rate"]:.0%} top-4 (n={int(best["games"])})')
    print()